# 버스 정류장 이벤트 데이터 전처리

이 노트북은 버스 정류장 이벤트 데이터를 불러와 전처리하고, 분석하기 쉬운 형태로 가공하는 과정을 담고 있습니다.

**주요 작업:**
1.  **데이터 로드**: 원본 데이터를 불러오고 기본 구조를 확인합니다.
2.  **시간대 보정**: 표준시(UTC)로 추정되는 시간을 한국 시간(KST, UTC+9)으로 변환합니다.
3.  **날짜 및 시간 데이터 분리**: 분석에 용이하도록 단일 시간 필드를 연, 월, 일, 시 등 여러 세부 필드로 분리합니다.
4.  **데이터 병합 및 저장**: 전처리된 데이터를 병합하여 새로운 CSV 파일로 저장합니다.
5.  **결과 확인**: 최종 결과물의 일부를 출력하여 작업이 올바르게 완료되었는지 검증합니다.

In [1]:
# 노트북 실행에 필요한 라이브러리(tqdm, pandas)를 설치합니다.
!pip3 install tqdm pandas

## 1. 환경 설정 및 데이터 로드

필요한 라이브러리를 설치하고, 원본 CSV 파일을 불러와 데이터프레임으로 변환합니다.
데이터의 기본적인 정보(`info()`)와 상위 5개 행(`head()`)을 출력하여 구조를 파악합니다.

In [2]:
import pandas as pd

# CSV 파일을 읽어와 masterDF(마스터 데이터프레임)에 저장합니다.
masterDF = pd.read_csv("./data/_gs_busevent.csv")
# 데이터프레임의 전체적인 정보(컬럼별 데이터 타입, null 값 유무 등)를 출력합니다.
masterDF.info()
# 데이터의 첫 5개 행을 출력하여 내용을 확인합니다.
masterDF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202935 entries, 0 to 202934
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   route_name         202935 non-null  object 
 1   bstop_arrive_time  202935 non-null  int64  
 2   bstop_leave_time   202935 non-null  int64  
 3   stop_time          202935 non-null  int64  
 4   node_name          202935 non-null  object 
 5   node_x_pos         202935 non-null  float64
 6   node_y_pos         202935 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 10.8+ MB


,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354


## 2. 시간대 보정 (UTC → KST)

`bstop_arrive_time`과 `bstop_leave_time` 필드는 UTC(협정 세계시) 기준으로 기록된 것으로 보입니다.
데이터의 시간 기준을 한국 표준시(KST, UTC+9)로 맞추기 위해 9시간을 더하는 작업을 수행합니다.

이 과정에서 `23:00`에 9시간을 더하면 `32:00`가 되는 대신, 날짜가 하루 넘어가고 시간이 `08:00`으로 올바르게 계산되도록 Pandas의 날짜/시간 처리 기능을 사용합니다.

In [3]:
# # 'bstop_arrive_time'과 'bstop_leave_time' 열을 문자열로 변환한 후, 날짜/시간 형식으로 파싱합니다.
# # format='%Y%m%d%H%M%S'는 '연도월일시분초' 형식의 문자열을 해석하는 규칙입니다.
# arrive_time_dt = pd.to_datetime(masterDF['bstop_arrive_time'].astype(str), format='%Y%m%d%H%M%S')
# leave_time_dt = pd.to_datetime(masterDF['bstop_leave_time'].astype(str), format='%Y%m%d%H%M%S')

# # 각 시간 데이터에 9시간을 더하여 한국 시간(KST)으로 보정합니다.
# # pd.Timedelta(hours=9)는 9시간의 시간 간격을 나타내는 객체입니다.
# # .dt.strftime('%Y%m%d%H%M%S')는 날짜/시간 객체를 다시 '연도월일시분초' 형식의 문자열로 변환합니다.
# masterDF['bstop_arrive_time'] = (arrive_time_dt + pd.Timedelta(hours=9)).dt.strftime('%Y%m%d%H%M%S')
# masterDF['bstop_leave_time'] = (leave_time_dt + pd.Timedelta(hours=9)).dt.strftime('%Y%m%d%H%M%S')

# # 문자열로 변환된 시간 데이터를 다시 64비트 정수형(int64)으로 변환하여 원래 데이터 타입을 유지합니다.
# masterDF['bstop_arrive_time'] = masterDF['bstop_arrive_time'].astype('int64')
# masterDF['bstop_leave_time'] = masterDF['bstop_leave_time'].astype('int64')

# # 처리된 후의 데이터 첫 5개 행을 확인합니다.
# masterDF.head()

In [4]:
masterDF.info()
masterDF.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202935 entries, 0 to 202934
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   route_name         202935 non-null  object 
 1   bstop_arrive_time  202935 non-null  int64  
 2   bstop_leave_time   202935 non-null  int64  
 3   stop_time          202935 non-null  int64  
 4   node_name          202935 non-null  object 
 5   node_x_pos         202935 non-null  float64
 6   node_y_pos         202935 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 10.8+ MB


,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354


## 3. 날짜 및 시간 데이터 분리

`YYYYMMDDHHMMSS` 형식의 단일 시간 필드를 분석에 용이하도록 연, 월, 일, 시, 분, 초 등 개별 필드로 분리합니다.
이 작업을 통해 특정 날짜나 시간대를 기준으로 데이터를 필터링하거나 집계하는 분석을 쉽게 수행할 수 있습니다.

In [5]:
bstop_arrive = []
bstop_leave = []

In [6]:
from datetime import datetime
from tqdm import tqdm

# 'bstop_arrive_time' 열의 각 시간 데이터에 대해 반복 작업을 수행합니다.
# tqdm은 진행 상태를 시각적으로 보여주는 라이브러리입니다.
for time in tqdm(masterDF['bstop_arrive_time']):
    # 시간 데이터를 문자열로 변환하고, datetime.strptime을 사용해 날짜/시간 객체로 파싱합니다.
    time_data = datetime.strptime(str(time), "%Y%m%d%H%M%S")

    # 파싱된 날짜/시간 객체에서 연, 월, 일, 시, 분, 초 등의 정보를 추출하여 사전에 저장합니다.
    bstop_arrive.append({
        "bstop_arrive_year":time_data.year, 
        "bstop_arrive_month":time_data.month, 
        "bstop_arrive_day":time_data.day, 
        "bstop_arrive_days":time_data.strftime("%Y-%m-%d"), 
        "bstop_arrive_hour":time_data.hour, 
        "bstop_arrive_minute":time_data.minute, 
        "bstop_arrive_second":time_data.second,
        "bstop_arrive_timezone": "UTC+9",
    })

100%|██████████| 202935/202935 [00:02<00:00, 78978.92it/s]


In [7]:
from datetime import datetime
from tqdm import tqdm

# 'bstop_leave_time' 열에 대해서도 도착 시간과 동일한 방식으로 시간 분리 작업을 수행합니다.
for time in tqdm(masterDF['bstop_leave_time']):
    time_data = datetime.strptime(str(time), "%Y%m%d%H%M%S")

    bstop_leave.append({
        "bstop_leave_year":time_data.year, 
        "bstop_leave_month":time_data.month, 
        "bstop_leave_day":time_data.day, 
        "bstop_leave_days":time_data.strftime("%Y-%m-%d"), 
        "bstop_leave_hour":time_data.hour, 
        "bstop_leave_minute":time_data.minute, 
        "bstop_leave_second":time_data.second,
        "bstop_leave_timezone": "UTC+9",
    })

100%|██████████| 202935/202935 [00:02<00:00, 83208.01it/s]


In [8]:
import pandas as pd

# 분리된 도착 및 출발 시간 데이터(리스트)를 각각 별도의 데이터프레임으로 변환합니다.
bstop_arrive = pd.DataFrame(bstop_arrive)
bstop_leave = pd.DataFrame(bstop_leave)

# 두 데이터프레임을 열(axis=1) 기준으로 합칩니다.
# join='inner'는 두 데이터프레임에 모두 존재하는 행만 유지하는 방식입니다.
bstop_time = pd.concat([bstop_arrive, bstop_leave], axis=1, join='inner')
bstop_time

,bstop_arrive_year,bstop_arrive_month,bstop_arrive_day,bstop_arrive_days,bstop_arrive_hour,bstop_arrive_minute,bstop_arrive_second,bstop_arrive_timezone,bstop_leave_year,bstop_leave_month,bstop_leave_day,bstop_leave_days,bstop_leave_hour,bstop_leave_minute,bstop_leave_second,bstop_leave_timezone
0,2020,1,4,2020-01-04,0,0,53,UTC+9,2020,1,4,2020-01-04,0,1,40,UTC+9
1,2020,1,4,2020-01-04,0,2,14,UTC+9,2020,1,4,2020-01-04,0,2,40,UTC+9
2,2020,1,4,2020-01-04,0,5,1,UTC+9,2020,1,4,2020-01-04,0,5,25,UTC+9
3,2020,1,4,2020-01-04,0,5,54,UTC+9,2020,1,4,2020-01-04,0,6,20,UTC+9
4,2020,1,4,2020-01-04,0,6,48,UTC+9,2020,1,4,2020-01-04,0,6,58,UTC+9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202930,2020,1,3,2020-01-03,23,55,11,UTC+9,2020,1,3,2020-01-03,23,55,17,UTC+9
202931,2020,1,3,2020-01-03,23,56,21,UTC+9,2020,1,3,2020-01-03,23,56,50,UTC+9
202932,2020,1,3,2020-01-03,23,57,0,UTC+9,2020,1,3,2020-01-03,23,57,9,UTC+9
202933,2020,1,3,2020-01-03,23,58,36,UTC+9,2020,1,3,2020-01-03,23,58,53,UTC+9


## 4. 데이터 병합 및 저장

분리된 날짜/시간 데이터를 원본 데이터프레임과 병합하여 분석 준비를 마칩니다.
이후 재사용을 위해 전처리된 전체 데이터를 `_gs_busevent_preprocessed.csv` 파일로 저장합니다.

In [9]:
import pandas as pd

# 분리된 시간 정보가 담긴 bstop_time 데이터프레임을 원본 masterDF에 추가로 병합합니다.
masterDF = pd.concat([masterDF, bstop_time], axis=1, join='inner')
masterDF

,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos,bstop_arrive_year,bstop_arrive_month,bstop_arrive_day,...,bstop_arrive_second,bstop_arrive_timezone,bstop_leave_year,bstop_leave_month,bstop_leave_day,bstop_leave_days,bstop_leave_hour,bstop_leave_minute,bstop_leave_second,bstop_leave_timezone
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229,2020,1,4,...,53,UTC+9,2020,1,4,2020-01-04,0,1,40,UTC+9
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398,2020,1,4,...,14,UTC+9,2020,1,4,2020-01-04,0,2,40,UTC+9
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077,2020,1,4,...,1,UTC+9,2020,1,4,2020-01-04,0,5,25,UTC+9
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677,2020,1,4,...,54,UTC+9,2020,1,4,2020-01-04,0,6,20,UTC+9
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354,2020,1,4,...,48,UTC+9,2020,1,4,2020-01-04,0,6,58,UTC+9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202930,818,20200103235511,20200103235517,6,은호1리국군대구병원 건너,128.78962,35.89662,2020,1,3,...,11,UTC+9,2020,1,3,2020-01-03,23,55,17,UTC+9
202931,818,20200103235621,20200103235650,30,경일·호산대학교(하양방향),128.80120,35.89989,2020,1,3,...,21,UTC+9,2020,1,3,2020-01-03,23,56,50,UTC+9
202932,818,20200103235700,20200103235709,9,부호리 건너,128.80327,35.90042,2020,1,3,...,0,UTC+9,2020,1,3,2020-01-03,23,57,9,UTC+9
202933,818,20200103235836,20200103235853,18,대구가톨릭대학교 건너,128.81219,35.90770,2020,1,3,...,36,UTC+9,2020,1,3,2020-01-03,23,58,53,UTC+9


In [10]:
import pandas as pd

# 최종적으로 전처리된 masterDF 데이터프레임을 CSV 파일로 저장합니다.
# index=False 옵션은 데이터프레임의 인덱스를 파일에 포함하지 않도록 설정합니다.
masterDF.to_csv("./data/_gs_busevent_preprocessed.csv", index=False)

## 5. 결과 확인

모든 전처리 과정이 끝난 후, 최종 결과물인 CSV 파일을 다시 불러와 상위 5개 행을 사전(dictionary) 형태로 출력합니다.
이를 통해 데이터가 의도한 대로 처리되었는지 최종적으로 검증합니다.

In [11]:
import pandas as pd

# 저장된 전처리 결과 CSV 파일을 다시 읽어옵니다.
# [0:5]를 통해 첫 5개 행만 선택하고, .to_dict()를 사용해 사전 형태로 변환하여 출력합니다.
# 이를 통해 데이터가 올바르게 저장되었는지 최종 확인합니다.
pd.read_csv("./data/_gs_busevent_preprocessed.csv")[0:5].to_dict()

{'route_name': {0: '818', 1: '818', 2: '818', 3: '818', 4: '818'},
 'bstop_arrive_time': {0: 20200104000053,
  1: 20200104000214,
  2: 20200104000501,
  3: 20200104000554,
  4: 20200104000648},
 'bstop_leave_time': {0: 20200104000140,
  1: 20200104000240,
  2: 20200104000525,
  3: 20200104000620,
  4: 20200104000658},
 'stop_time': {0: 49, 1: 27, 2: 25, 3: 27, 4: 11},
 'node_name': {0: '하양초등학교',
  1: '하양대구은행',
  2: '금락초교 건너',
  3: '부기리LH아파트',
  4: '부기2리'},
 'node_x_pos': {0: 128.81746,
  1: 128.8208,
  2: 128.82316,
  3: 128.82388,
  4: 128.8243},
 'node_y_pos': {0: 35.91229,
  1: 35.91398,
  2: 35.91077,
  3: 35.90677,
  4: 35.90354},
 'bstop_arrive_year': {0: 2020, 1: 2020, 2: 2020, 3: 2020, 4: 2020},
 'bstop_arrive_month': {0: 1, 1: 1, 2: 1, 3: 1, 4: 1},
 'bstop_arrive_day': {0: 4, 1: 4, 2: 4, 3: 4, 4: 4},
 'bstop_arrive_days': {0: '2020-01-04',
  1: '2020-01-04',
  2: '2020-01-04',
  3: '2020-01-04',
  4: '2020-01-04'},
 'bstop_arrive_hour': {0: 0, 1: 0, 2: 0, 3: 0, 4: 0},
 'bstop_